# 🐍 Day 7: Mini-Project - Async Web Scraper

- Async HTTP with aiohttp
- Concurrent URL fetching
- Rate limiting with Semaphore
- Complete runnable scraper

## Project Overview

We'll build an async web scraper that:
- Fetches multiple URLs concurrently
- Uses a Semaphore to limit concurrent connections
- Parses response (title/length)
- Handles errors gracefully

**Note**: Install aiohttp: `pip install aiohttp`

## Step 1: Basic Fetch

Fetch a single URL with aiohttp.

In [ ]:
import asyncio

async def fetch_url(session, url):
    try:
        import aiohttp
        async with session.get(url, timeout=aiohttp.ClientTimeout(total=10)) as response:
            text = await response.text()
            return {"url": url, "status": response.status, "length": len(text)}
    except Exception as e:
        return {"url": url, "error": str(e)}

async def main():
    try:
        import aiohttp
        async with aiohttp.ClientSession() as session:
            result = await fetch_url(session, "https://httpbin.org/get")
            print(result)
    except ImportError:
        print("pip install aiohttp")

asyncio.run(main())

## Step 2: Fetch Multiple URLs

Use gather to fetch several URLs concurrently.

In [ ]:
import asyncio

async def fetch_url(session, url):
    try:
        import aiohttp
        async with session.get(url, timeout=aiohttp.ClientTimeout(total=10)) as response:
            text = await response.text()
            return {"url": url, "status": response.status, "length": len(text)}
    except Exception as e:
        return {"url": url, "error": str(e)}

async def fetch_all(session, urls):
    tasks = [fetch_url(session, url) for url in urls]
    return await asyncio.gather(*tasks)

async def main():
    urls = [
        "https://httpbin.org/get",
        "https://httpbin.org/delay/0",
        "https://httpbin.org/bytes/100",
    ]
    try:
        import aiohttp
        async with aiohttp.ClientSession() as session:
            results = await fetch_all(session, urls)
            for r in results:
                print(r)
    except ImportError:
        print("pip install aiohttp")

asyncio.run(main())

## Step 3: Add Rate Limiting with Semaphore

Limit concurrent requests to avoid overwhelming servers.

In [ ]:
import asyncio

async def fetch_url(session, semaphore, url):
    async with semaphore:
        try:
            import aiohttp
            async with session.get(url, timeout=aiohttp.ClientTimeout(total=10)) as response:
                text = await response.text()
                return {"url": url, "status": response.status, "length": len(text)}
        except Exception as e:
            return {"url": url, "error": str(e)}

async def fetch_all(session, urls, max_concurrent=3):
    semaphore = asyncio.Semaphore(max_concurrent)
    tasks = [fetch_url(session, semaphore, url) for url in urls]
    return await asyncio.gather(*tasks)

async def main():
    urls = ["https://httpbin.org/get"] * 5
    try:
        import aiohttp
        async with aiohttp.ClientSession() as session:
            results = await fetch_all(session, urls, max_concurrent=2)
            for r in results:
                print(r["url"], r.get("status", r.get("error")))
    except ImportError:
        print("pip install aiohttp")

asyncio.run(main())

## Step 4: Extract Title (Simple HTML Parse)

Use regex to extract <title> from HTML.

In [ ]:
import re

def extract_title(html):
    match = re.search(r'<title[^>]*>([^<]+)</title>', html, re.IGNORECASE)
    return match.group(1).strip() if match else "(no title)"

html = "<html><head><title>My Page</title></head><body></body></html>"
print(extract_title(html))

## Step 5: Complete Scraper Class

Wrap everything in a Scraper class.

In [ ]:
import asyncio
import re

def extract_title(html):
    match = re.search(r'<title[^>]*>([^<]+)</title>', html, re.IGNORECASE)
    return match.group(1).strip() if match else "(no title)"

class AsyncScraper:
    def __init__(self, max_concurrent=5, timeout=10):
        self.max_concurrent = max_concurrent
        self.timeout = timeout

    async def fetch(self, session, semaphore, url):
        async with semaphore:
            try:
                import aiohttp
                async with session.get(url, timeout=aiohttp.ClientTimeout(total=self.timeout)) as resp:
                    text = await resp.text()
                    return {
                        "url": url,
                        "status": resp.status,
                        "length": len(text),
                        "title": extract_title(text),
                    }
            except Exception as e:
                return {"url": url, "error": str(e)}

    async def scrape(self, urls):
        try:
            import aiohttp
        except ImportError:
            raise ImportError("pip install aiohttp")
        import aiohttp
        sem = asyncio.Semaphore(self.max_concurrent)
        async with aiohttp.ClientSession() as session:
            tasks = [self.fetch(session, sem, url) for url in urls]
            return await asyncio.gather(*tasks)

## Step 6: Run the Scraper

Demo with real URLs.

In [ ]:
import asyncio
import time

async def main():
    urls = [
        "https://httpbin.org/html",
        "https://httpbin.org/get",
        "https://httpbin.org/delay/0",
    ]
    scraper = AsyncScraper(max_concurrent=3)
    start = time.perf_counter()
    results = await scraper.scrape(urls)
    elapsed = time.perf_counter() - start
    for r in results:
        print(r)
    print(f"\nFetched {len(results)} URLs in {elapsed:.2f}s")

asyncio.run(main())

## Step 7: Full Code (Fallback for No aiohttp)

If aiohttp isn't installed, use simulated fetches.

In [ ]:
import asyncio
import re

def extract_title(html):
    match = re.search(r'<title[^>]*>([^<]+)</title>', html, re.IGNORECASE)
    return match.group(1).strip() if match else "(no title)"

async def scrape_urls(urls, max_concurrent=5):
    try:
        import aiohttp
        sem = asyncio.Semaphore(max_concurrent)
        async def fetch(session, url):
            async with sem:
                async with session.get(url, timeout=aiohttp.ClientTimeout(total=10)) as r:
                    text = await r.text()
                    return {"url": url, "status": r.status, "length": len(text), "title": extract_title(text)}
        async with aiohttp.ClientSession() as session:
            return await asyncio.gather(*[fetch(session, u) for u in urls])
    except ImportError:
        # Simulate without aiohttp
        async def sim_fetch(url):
            await asyncio.sleep(0.2)
            return {"url": url, "status": 200, "length": 100, "title": "(simulated)"}
        return await asyncio.gather(*[sim_fetch(u) for u in urls])

results = asyncio.run(scrape_urls(["https://example.com", "https://httpbin.org/html"]))
for r in results:
    print(r)

## Extensions to Try

- Add retry logic with exponential backoff
- Use BeautifulSoup for better HTML parsing
- Save results to JSON/CSV
- Add User-Agent header
- Respect robots.txt

## Key Takeaways

- aiohttp for async HTTP
- Semaphore for rate limiting
- gather for concurrent fetches
- Handle errors per-URL

## 🎉 Week 11 Complete!

You've mastered threading, multiprocessing, the GIL, and asyncio. Next week: design patterns and performance!